# Z639 Topic Modeling Project – Annotated Notebook
This notebook walks you step-by-step through running BERTopic.

In [1]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

!pip install bertopic
!pip install sentence-transformers

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

RANDOM_STATE = 42

  Using cached hdbscan-0.8.41-cp311-cp311-macosx_10_9_universal2.whl.metadata (15 kB)
  Using cached umap_learn-0.5.11-py3-none-any.whl.metadata (26 kB)
  Using cached plotly-6.6.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached llvmlite-0.46.0.tar.gz (193 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numba-0.64.0-cp311-cp311-macosx_10_15_x86_64.whl
  Using cached pynndescent-0.6.0-py3-none-any.whl.metadata (6.9 kB)
Using cached hdbscan-0.8.41-cp311-cp311-macosx_10_9_universal2.whl (1.4 MB)
Using cached plotly-6.6.0-py3-none-any.whl (9.9 MB)
Using cached umap_learn-0.5.11-py3-none-any.whl (90 kB)
Using cached pynndescent-0.6.0-py3-none-any.whl (73 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for llvmlite (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [42 lines of output]
      /private/var/folders/gz/khf86byn4hb441skkcdc1x380000gn/T/pip-bui

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/chrisokoro/opt/anaconda3/envs/python311/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/chrisokoro/opt/anaconda3/envs/python311/lib/python3.11/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/Users/chrisokoro/opt/anaconda3/envs/p

NameError: name 'nn' is not defined

## 1. Load Dataset
The dataset (text_data.csv) must be located in the same folder as the current notebook.

In [ ]:
df_base = pd.read_csv('test.csv')
# df_base = pd.read_excel('paper3data.xlsx')
df_base["doc_id"] = df_base.index



df = df_base.sample(2500, random_state=RANDOM_STATE)
df.head()
df2 = df.copy()

In [ ]:
df_base.shape

## 2. Minimal Cleaning
We remove URLs and lowercase to reduce noise for the embedding model.

In [ ]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df['clean_text'] = df['text'].apply(clean)
docs = df['clean_text'].tolist()

## 3. Build BERTopic Model
We embed documents using `all-MiniLM-L6-v2`, then cluster with BERTopic.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(docs, show_progress_bar=True)

topic_model = BERTopic(embedding_model=model, calculate_probabilities=True)
topics, probs = topic_model.fit_transform(docs, embeddings)

df['topic'] = topics
df['probability'] = probs.max(axis=1)
df.head()

## 4. Topic Info & Export


In [ ]:
info = topic_model.get_topic_info()
info.to_csv('topic_info.csv', index=False)
df[['doc_id','clean_text','topic','probability']].to_csv('topic_assignments.csv', index=False)
info.head()

## 5. Topic Frequency Plot
Use this figure in your report.

In [ ]:
subset = info[info['Topic'] >= 0]
plt.figure(figsize=(10,4))
plt.bar(subset['Topic'].astype(str), subset['Count'])
plt.xticks(rotation=45)
plt.title('Topic Frequencies')
plt.show()

In [ ]:
import re
import html

CONTRACTIONS = {
    "can't": "cannot", "won't": "will not", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
    "'t": " not", "'ve": " have", "'m": " am"
}

def expand_contractions(text: str) -> str:
    # lightweight contraction handling
    for k, v in CONTRACTIONS.items():
        text = text.replace(k, v)
    return text

def clean_text_v1(text):
    text = "" if text is None else str(text)
    text = html.unescape(text)                    # decode HTML entities
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)        # IMDB line breaks
    text = re.sub(r"<.*?>", " ", text)            # any remaining HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text) # URLs
    text = expand_contractions(text)
    text = re.sub(r"[^a-z\s]", " ", text)         # keep letters/spaces only
    text = re.sub(r"\s+", " ", text).strip()      # normalize whitespace
    return text

# df_base2 = df_base.copy()
# df_new = df_base2.sample(2500, random_state=RANDOM_STATE)

df2["clean_text"] = df2["text"].apply(clean_text_v1)
docs_new = df2["clean_text"].tolist()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Build embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(docs_new, show_progress_bar=True)

# Custom vectorizer for better topic words
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10
)

# Build & fit BERTopic
topic_model = BERTopic(
    embedding_model=model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs_new, embeddings)

# Attach results back to df
df2["topic"] = topics
df2["probability"] = probs.max(axis=1)
df2.head()

In [ ]:
info_new = topic_model.get_topic_info()
info_new.to_csv('topic_info_new.csv', index=False)
df2[['doc_id','clean_text','topic','probability']].to_csv('topic_assignments_new.csv', index=False)
info_new.head()

In [ ]:
subset = info_new[info_new['Topic'] >= 0]
plt.figure(figsize=(10,4))
plt.bar(subset['Topic'].astype(str), subset['Count'])
plt.xticks(rotation=45)
plt.title('Topic Frequencies')
plt.show()

In [ ]:

# Set option to display full width of columns (unlimited width)
pd.set_option('display.max_colwidth', None)

# Your DataFrame and .head() call
# print(df.head())


In [ ]:
info_new = topic_model.get_topic_info()
info_new.head(7)

In [ ]:
topic_model.get_topic(1)

In [ ]:
topic_model.get_topic(2)

In [ ]:
df2[df2["topic"] == 1][["clean_text"]].head(2)

In [ ]:
# Reset to default settings
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')


In [ ]:
topic_model.get_topic(0)

In [ ]:
info.head(15)

In [ ]:

info_new.head(15)

In [ ]:
pd.set_option('display.max_colwidth', None)
df2[df2["topic"] == -1][["clean_text"]].head(15)


In [ ]:
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')

In [ ]:

subset = info_new[info_new['Topic'] >= 0]
plt.figure(figsize=(10,4))
plt.bar(subset['Topic'].astype(str), subset['Count'])
plt.xticks(rotation=45)
plt.title('Topic Frequencies')
plt.xlabel("Topic ID")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
top_n = 15

subset = (
    info_new[info_new["Topic"] >= 0]
    .sort_values("Count", ascending=False)
    .head(top_n)
)

# Create color list: top 5 highlighted
colors = ["tab:blue" if i < 5 else "lightgray" for i in range(len(subset))]

plt.figure(figsize=(10, 5))
plt.bar(subset["Topic"].astype(str), subset["Count"], color=colors)

plt.xlabel("Topic ID")
plt.ylabel("Number of Reviews")
plt.title("Top 15 Most Frequent Topics (Top 5 Highlighted)")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
top_n = 15

subset = (
    info_new[info_new["Topic"] >= 0]
    .sort_values("Count", ascending=False)
    .head(top_n)
)

# Shorten long topic names for readability
subset["ShortName"] = subset["Name"].str.replace("_", " ").str[:30]

# Highlight top 5 topics
colors = ["tab:blue" if i < 5 else "lightgray" for i in range(len(subset))]

plt.figure(figsize=(12, 7))
plt.bar(subset["ShortName"], subset["Count"], color=colors)

plt.xlabel("Topic")
plt.ylabel("Number of Reviews")
plt.title("Top 15 Most Frequent Topics (Top 5 Highlighted)")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()